In [ ]:
!pip install 'powerlaw'

In [ ]:
import bmll2 as b2
from bmll2 import reference, Security, NormalisedSecurity, SparkHelper, get_market_data, get_market_data_range, VenueMarketError, get_market_tables, save_spark_dataframe, load_spark_dataframe

b2.get_file('modules/metaorder.py')
b2.get_file('modules/LMF.py')

import metaorder as metaorder
import LMF as LMF

import random
import math
import pandas as pd
import polars as pl
import numpy as np
from pandas import StringDtype
from datetime import timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import LogFormatterSciNotation

from statsmodels.sandbox.stats.runs import runstest_1samp
import powerlaw
import itertools
import pylab
import scipy.stats
import scipy.fft
from scipy.optimize import curve_fit
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.stattools import acf
from scipy.stats import t

from typing import Literal

In [ ]:
%%time
trader_participation_distribution = 'power'
exchange = 'XJSE'
files = b2.list_files(path = f'data/grid_search_7/{exchange}/{trader_participation_distribution}')

for f in files:
    b2.get_file(f'data/grid_search_7/{exchange}/{trader_participation_distribution}/{f}')

In [ ]:
data = pl.scan_parquet(f'*{trader_participation_distribution}_grid_search.parquet').collect()

if trader_participation_distribution == 'homogenous':
    data.sort(['ticker', 'year', 'N'])
    data.head(10)
if trader_participation_distribution == 'power':
    data.sort(['ticker', 'year', 'N', 'delta'])
    data.head(10)

In [ ]:
# estimate = the estimated value
# expected_value = the expectd_value for the point estimate
# var = the variance of the point estimate
# lambda_penalty = the penalty term that the variance should be multiplied by

def error_calculation(estimate, expected_value, var, lambda_penalty = 1):
    error = np.abs(estimate - expected_value) / np.abs(expected_value) 
    ans   = error + lambda_penalty * (var / expected_value ** 2)
    return ans

In [ ]:
# returns a dataframe that contains the best configuration for each time scale
# data = dataframe containing the configurations details with estimates for the exponents with variances and interval half widths
# config = do you want the configuration that fits the microscopic/macroscopic values the best. Microscopic refers to the impact curve exponent
# the phi exponent and beta
# estimation_method = wether you want the results for when gamma is estimated using the NLLS or the PSD method
# lambda_penatly = the penalty term for the variance
# mix = how the ratio that individual errors for the impact curve exponent, exponent for phi, and beta should be mixed to get the total macroscopic error

def find_configs(data, trader_participation_distribution, config = Literal['metaorder', 'LMF', 'N', 'combined'],
                   estimation_method = Literal['NLLS', 'PSD'], lambda_penalty = 1, mix = [1, 1, 1]):

    impact_error = error_calculation(data[:, 'impact curve exponent'], 0.5, data[:, 'impact curve exponent var'], lambda_penalty)
    phi_error    = error_calculation(data[:, 'phi exponent'], 0.5, data[:, 'phi exponent var'], lambda_penalty)
    gamma        = data[:, f'gamma ({estimation_method.lower()})']
    beta_error   = error_calculation(data[:, 'beta'], 0.5 * (1 - gamma), data[:, 'beta var'], lambda_penalty)
    alpha_error  = np.abs(data[:, 'alpha'] - 1 - gamma) / (gamma + 1)
    N_error      = np.abs(data[:, 'N (st)'] - data[:, f'N ({estimation_method.lower()})']) / data[:, 'N (st)']

    if trader_participation_distribution == 'homogenous':
        error_df = pl.DataFrame({'year': data.select('year'), 'ticker': data.select('ticker'), 'N': data.select('N'),
                                 f'N_{{ST}}': data.select('N (st)'),
                                 f'N_{{est}}': data.select(f'N ({estimation_method.lower()})'),
                                 #f'$\\delta$': data.select('delta'), 
                                 f'$\\alpha$': data.select('alpha'), 
                                 f'$\\gamma$': data.select(f'gamma ({estimation_method.lower()})'),
                                 
                                 f'\\makecell{{$\\frac{{Q}}{{V_D}}$ \\\\ exponent}}': data.select('impact curve exponent'),
                                 f'\\makecell{{$\\phi$ \\\\ exponent}}': data.select('phi exponent'), 
                                 f'$\\beta$': data.select('beta'), 
                                 f'$\\mathrm{{e}}_{{M}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix),
                                 f'$\\mathrm{{e}}_{{LMF}}$ ({estimation_method})': alpha_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': N_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix) + alpha_error})
    
    if trader_participation_distribution == 'power':
        error_df = pl.DataFrame({'year': data.select('year'), 'ticker': data.select('ticker'), 'N': data.select('N'),
                                 f'N_{{ST}}': data.select('N (st)'),
                                 f'N_{{est}}': data.select(f'N ({estimation_method.lower()})'),
                                 f'$\\delta$': data.select('delta'), 
                                 f'$\\alpha$': data.select('alpha'), 
                                 f'$\\gamma$': data.select(f'gamma ({estimation_method.lower()})'),
                                 
                                 f'\\makecell{{$\\frac{{Q}}{{V_D}}$ \\\\ exponent}}': data.select('impact curve exponent'),
                                 f'\\makecell{{$\\phi$ \\\\ exponent}}': data.select('phi exponent'), 
                                 f'$\\beta$': data.select('beta'), 
                                 f'$\\mathrm{{e}}_{{M}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix),
                                 f'$\\mathrm{{e}}_{{LMF}}$ ({estimation_method})': alpha_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': N_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix) + alpha_error})
        
    if (config == 'metaorder'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{M}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df

    if (config == 'LMF'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{LMF}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df

    if (config == 'N'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df

    if (config == 'combined'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df
        

In [ ]:
trader_participation_distribution = 'power'
config = 'metaorder'
exchange = 'XJSE'
files = b2.list_files(path = f'data/metaorders/{exchange}/calibrated/{trader_participation_distribution}/{config}')

for f in files:
    b2.get_file(f'data/metaorders/{exchange}/calibrated/{trader_participation_distribution}/{config}/{f}')

In [ ]:
all_metaorders = pl.scan_parquet(f'*_XJSE_power_{config}_metaorder_data_calibrated.parquet').collect()

all_metaorders.head(10)

In [ ]:
%%time
#metaorders = metaorder.metaorder_data(stock, group = group, N = N, trader_participation_distribution = trader_participation_distribution, delta = delta)
executions = metaorder.execution_data(stock, group = group, N = N, trader_participation_distribution = trader_participation_distribution, delta = delta, min_num_child_orders = 5)
#decays     = metaorder.decay_data(stock, group = group, N = N, trader_participation_distribution = trader_participation_distribution, delta = delta, min_num_child_orders = 5)

In [ ]:
b2.create_folder('figures_2', ensure_exists = True)
b2.create_folder('figures_2/concave_profile', ensure_exists = True)
b2.create_folder(f'figures_2/concave_profile/{exchange}', ensure_exists = True) 
b2.create_folder(f'figures_2/concave_profile/{exchange}/calibrated', ensure_exists = True) 
b2.create_folder(f'figures_2/concave_profile/{exchange}/calibrated/{config}', ensure_exists = True)

In [ ]:
bins = np.linspace(0, 1, 51)
bin_mids = (bins[:-1] + bins[1:]) / 2

df        = executions.with_columns(pl.col('phi').cut(bins).alias('bin'))
binned    = df.group_by('bin').agg(pl.col('dynamical metaorder impact').mean().alias('dynamical metaorder impact (binned)'))
full_bins = pl.DataFrame({'phi': bin_mids}).with_columns(pl.col('phi').cut(bins).alias('bin'))
df        = full_bins.join(binned, on = 'bin', how = 'left').sort('phi')

scaled_impact_binned = df['dynamical metaorder impact (binned)']

if len(scaled_impact_binned) < 10:
    print('error')
    
scaled_impact_binned = pl.concat([pl.Series([0.0]), scaled_impact_binned])
bin_mids             = np.insert(bin_mids, 0, 0)
phi                  = bin_mids

def execution_profile_model(phi, exponent, Y):
    return  Y * (phi ** exponent)

phi    = np.array(phi)
impact = np.array(scaled_impact_binned)

mask       = (phi > 0) & np.isfinite(impact)
phi_fit    = phi[mask]
impact_fit = impact[mask]

if len(phi_fit) < 10 or len(impact_fit) < 10:
    print('error')

try:
    params, covariance = curve_fit(execution_profile_model, phi_fit, impact_fit, p0 = [0.5, 1], maxfev = 100000)

except RuntimeError:
    print('error')
    
phi_est      = params[0]
Y_est        = params[1]

fitted_curve = execution_profile_model(bin_mids, phi_est, Y_est)

n   = len(phi_fit)
p   = len(params)          # number of parameters (2 here)
dof = n - p                # degrees of freedom

alpha = 0.05               # 95% CI
tval  = t.ppf(1 - alpha / 2, dof)
se    = np.sqrt(np.diag(covariance))
interval_half_widths = tval * se[0], tval * se[1]

print(f'Y: {Y_est:.3f} \\pm {se[1]:.3f}')
print(f'exponent: {phi_est:.3f} \\pm {se[0]:.3f}')
print(f'{ticker} & {Y_est:.3f} $\\pm$ {se[1]:.3f} & {phi_est:.3f} $\\pm$ {se[0]:.3f}')

fig, ax = plt.subplots(figsize = (6, 6))

ax.scatter(bin_mids, scaled_impact_binned, color = col, marker = 'o', label = f'{ticker} dynamic impact')
ax.set_xlabel(r'Rescaled volume time $\phi = \sum q_i/Q$', fontsize = 14)
ax.set_ylabel(r'Dynamic impact $\frac{I(\phi Q)}{\sigma_{D} \sqrt{Q/V_{D}}}$', fontsize = 14)
ax.plot(bin_mids, fitted_curve, color = 'red', linestyle = '--', label = rf'Fitted curve: $Y \times\phi^{{{phi_est:.2f}}}$')
ax.set_yscale('linear')
ax.legend()

ax.set_box_aspect(1)
plt.tight_layout()

#plt.savefig(f'{ticker}_{exchange}_{config}_calibrated_concave_profile.pdf')
#b2.put_file(f'{ticker}_{exchange}_{config}_calibrated_concave_profile.pdf', f'figures_2/concave_profile/{exchange}/calibrated/{config}')
#b2.get_file(f'figures_2/concave_profile/{exchange}/calibrated/{config}/{ticker}_{exchange}_{config}_calibrated_concave_profile.pdf')

plt.show()
